# **Configurações**

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS fiap

In [0]:

import os
import gc
import pandas as pd

from pyspark.sql.functions import col, to_date, when, substring, date_format, expr, sum, count

ANO = 2025
MES = list(range(1, 13))
UF = "SP"
CATALOGO = "fiap" 

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.gold")

# **Ingestão IBGE**


## **BRONZE**

In [0]:
import requests
import pandas as pd
from pyspark.sql.functions import current_timestamp

# via API do IBGE
#  SP 35
url = "https://servicodados.ibge.gov.br/api/v1/localidades/estados/35/municipios"

response = requests.get(url)
if response.status_code == 200:
    data = response.json()
    
    pdf_ibge = pd.json_normalize(data)
    
    #padroniza colunas 
    pdf_ibge.columns = [c.replace(".", "_").upper() for c in pdf_ibge.columns]
    
    #converte para Spark
    df_ibge_bronze = spark.createDataFrame(pdf_ibge.astype(str))
    
    #metadados
    df_ibge_bronze = df_ibge_bronze.withColumn("DATA_INGESTAO", current_timestamp())
    
    #salvando na Bronze
    df_ibge_bronze.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOGO}.bronze.IBGE_SP")
    print(f"Bronze IBGE criada com {df_ibge_bronze.count()} municípios de SP")
else:
    print("Erro ao acessar API do IBGE")

## **SILVER**

In [0]:
from pyspark.sql.functions import col, substring

df_bronze = spark.table(f"{CATALOGO}.bronze.IBGE_SP")

df_ibge_silver = df_bronze.select(
    col("ID").alias("COD_IBGE_COMPLETO"),                 # 7 dígitos (Oficial)
    substring(col("ID"), 1, 6).alias("COD_SUS"), # 6 dígitos (qnd eu for cruzar c SINAN)
    col("NOME").alias("NOME_MUNICIPIO"),
    col("MICRORREGIAO_NOME").alias("NOME_MICRORREGIAO"),
    col("MICRORREGIAO_MESORREGIAO_NOME").alias("NOME_MESORREGIAO") 
)

# Salva na Silver
df_ibge_silver.write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable(f"{CATALOGO}.silver.IBGE_SP")

print(f" Silver IBGE atualizada,{CATALOGO}.silver.IBGE_SP ")

In [0]:
%sql
SELECT * FROM FIAP.SILVER.ibge_sp